# Adult Income 소득 예측 (>50K)
1994년 미국 인구조사 데이터로 연소득 5만 달러 초과 여부를 예측합니다.

**결과:** Test 정확도 약 0.873 (기준선 0.764)

## 1. 데이터 불러오기
코랩에서는 아래 셀이 저장소를 받아옵니다. 로컬에서 실행할 때는 건너뛰세요.

In [ ]:
import os
if not os.path.exists('data'):
    !git clone https://github.com/<YOUR_ID>/adult-income.git
    %cd adult-income

In [ ]:
import numpy as np
import pandas as pd

COLS = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

# 헤더 없음, ", " 구분, 결측치는 "?"
train = pd.read_csv("data/raw/adult-training.csv", header=None, names=COLS,
                    skipinitialspace=True, na_values="?")
test = pd.read_csv("data/raw/adult-test.csv", header=None, names=COLS, skiprows=1,  # 첫 줄 쓰레기 행
                   skipinitialspace=True, na_values="?")
print(train.shape, test.shape)
train.head()

## 2. 전처리
| 처리 | 이유 |
|---|---|
| `fnlwgt` 삭제 | 인구조사 표본 가중치. 개인 특성이 아니고 소득과의 상관 ≈ 0 |
| `education` 삭제 | `education_num`과 1:1 대응하는 중복 정보 |
| 결측치 `?` → `Unknown` | 결측 행이 7.4%. 결측 자체가 정보(결측 그룹의 고소득 비율은 10%, 평균은 25%) |
| `native_country` 묶기 | 90%가 미국. 미국 / 기타 / Unknown 3개로 |
| `capital_gain/loss`에 log1p | 90% 이상이 0인 극단적 치우침 |
| test 레이블의 `.` 제거 | `>50K.` → `>50K` |
| train 중복 행 제거 | 완전 중복 24행 |

In [ ]:
train = train.drop_duplicates()

def clean(df):
    df = df.copy()
    df["income"] = df["income"].str.rstrip(".").map({"<=50K": 0, ">50K": 1})
    for c in ["workclass", "occupation", "native_country"]:
        df[c] = df[c].fillna("Unknown")
    df["native_country"] = np.where(
        df["native_country"].isin(["United-States", "Unknown"]),
        df["native_country"], "Other")
    df["capital_gain"] = np.log1p(df["capital_gain"])
    df["capital_loss"] = np.log1p(df["capital_loss"])
    return df.drop(columns=["fnlwgt", "education"])

train, test = clean(train), clean(test)

cat_cols = ["workclass", "marital_status", "occupation", "relationship",
            "race", "sex", "native_country"]
X_train = pd.get_dummies(train.drop(columns="income"), columns=cat_cols, dtype=int)
X_test = pd.get_dummies(test.drop(columns="income"), columns=cat_cols, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train, y_test = train["income"], test["income"]
print(X_train.shape, X_test.shape)

## 3. 모델 학습 및 평가

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

model = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("기준선(전부 <=50K):", round(1 - y_test.mean(), 4))
print("Test 정확도:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=["<=50K", ">50K"]))

## 4. 모델 비교

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=0),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"{name:22s} {accuracy_score(y_test, m.predict(X_test)):.4f}")